## Imports e Configurações

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona a pasta src ao path
sys.path.append(os.path.abspath(os.path.join('..')))

from src.dataset import get_svhn_loaders
from src.model import SVHNNet
from src.train_utils import train_one_epoch, evaluate

# Configuração de dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando device: {device}")

# Hiperparâmetros Fixos para comparação justa
BATCH_SIZE = 64
EPOCHS = 10  # SVHN converge rápido, 10 é suficiente para ver diferenças
LR_SGD = 0.01 # Learning Rate padrão para SGD
LR_ADAM = 0.001 # Adam geralmente precisa de LR menor

# Carregar Dados
train_loader, test_loader = get_svhn_loaders(batch_size=BATCH_SIZE)

## Função de Experimento
Esta função é crucial. Ela garante que, para cada otimizador, você comece com um modelo "zerado" (pesos reinicializados). Se você não fizer isso, o segundo otimizador continuaria o treino do primeiro!

In [ ]:
def run_experiment(optimizer_name, optimizer_params, model_class, train_loader, test_loader, epochs):
    """
    Cria um modelo novo, configura o otimizador e treina.
    Retorna históricos de loss e acurácia.
    """
    print(f"--- Iniciando Experimento: {optimizer_name} ---")
    
    # 1. Instanciar novo modelo (Resetar pesos)
    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    
    # 2. Configurar Otimizador com base no nome
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), **optimizer_params)
    elif optimizer_name == 'SGD + Momentum':
        optimizer = optim.SGD(model.parameters(), **optimizer_params)
    elif optimizer_name == 'SGD + Nesterov':
        optimizer = optim.SGD(model.parameters(), **optimizer_params)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), **optimizer_params)
    
    # 3. Loop de Treino
    train_losses = []
    val_accuracies = []
    
    for epoch in range(epochs):
        t_loss, _ = train_one_epoch(model, train_loader, optimizer, criterion, device)
        _, v_acc = evaluate(model, test_loader, criterion, device)
        
        train_losses.append(t_loss)
        val_accuracies.append(v_acc)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {t_loss:.4f} | Val Acc: {v_acc:.2f}%")
        
    return train_losses, val_accuracies

## Execução dos Experimentos (SGD e Variantes)
Aqui cobrimos o Tópico 4 (SGD Simples, Momentum, Nesterov).

In [ ]:
results = {}

# 1. SGD Vanilla (Puro)
# Geralmente é lento e oscila muito ou fica preso em platôs
results['SGD'] = run_experiment(
    'SGD', 
    {'lr': LR_SGD}, 
    SVHNNet, train_loader, test_loader, EPOCHS
)

# 2. SGD com Momentum
# O Momentum (geralmente 0.9) ajuda a acumular velocidade na direção certa
results['SGD + Momentum'] = run_experiment(
    'SGD + Momentum', 
    {'lr': LR_SGD, 'momentum': 0.9}, 
    SVHNNet, train_loader, test_loader, EPOCHS
)

# 3. SGD com Nesterov
# O Nesterov calcula o gradiente "à frente" da posição atual. Mais estável teoricamente.
results['SGD + Nesterov'] = run_experiment(
    'SGD + Nesterov', 
    {'lr': LR_SGD, 'momentum': 0.9, 'nesterov': True}, 
    SVHNNet, train_loader, test_loader, EPOCHS
)

## Execução do Adam
Aqui cobrimos o Tópico 2 (Adam). Note que usamos uma Learning Rate diferente (geralmente 1e-3), pois o Adam se comporta de forma distinta do SGD.

In [ ]:
# 4. Adam
# Combina Momentum + RMSProp (escala adaptativa).
# Geralmente o campeão de convergência inicial.
results['Adam'] = run_experiment(
    'Adam', 
    {'lr': LR_ADAM}, 
    SVHNNet, train_loader, test_loader, EPOCHS
)

## Visualização Comparativa e Análise
Gera os gráficos obrigatórios para a Nota Técnica.

In [ ]:
# Configuração dos plots
plt.figure(figsize=(14, 6))

# Plot 1: Curva de Loss (Treino) - "Velocidade de Convergência"
plt.subplot(1, 2, 1)
for name, (losses, accs) in results.items():
    plt.plot(losses, label=name, marker='.')

plt.title('Comparação: Curva de Perda (Loss)')
plt.xlabel('Épocas')
plt.ylabel('Loss (CrossEntropy)')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Acurácia de Validação - "Desempenho Final"
plt.subplot(1, 2, 2)
for name, (losses, accs) in results.items():
    plt.plot(accs, label=name, marker='.')

plt.title('Comparação: Acurácia de Validação')
plt.xlabel('Épocas')
plt.ylabel('Acurácia (%)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()